# ML-02 — Research Question and Provisional Lane

## 1. My lane (or freestyle) and why

**Provisional lane: Refresh / Content Opportunity Scoring.**

I want to investigate how to prioritize content pages for review or refresh. This lane fits the starter data because each row represents a content item and includes observable search-performance and content-lifecycle signals such as impressions, CTR, position, content age, and days since the last update. The goal is not simply to train a model; it is to improve the decision of **which page should be reviewed first**. I will use the starter dataset for the initial evidence, then refine the question and, if useful, move to the warehouse data. This is a provisional choice and can change as the evidence develops.


In [1]:
# Load the starter dataset and confirm its size.
from pathlib import Path
import pandas as pd

candidates = [Path("data/raw/content_refresh_anonymized.csv"), Path("../data/raw/content_refresh_anonymized.csv")]
DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print(f"Starter dataset: {len(df):,} rows × {df.shape[1]} columns")


Starter dataset: 30,000 rows × 44 columns


## 2. The question: decision, action, cost of a wrong call

**Research question:** Given the observable information available for a content page, can I build a defensible score that helps an SEO/content team decide **which pages to review or refresh first**?

- **Decision:** Which pages belong near the top of the review queue?
- **Who acts:** An SEO or content editor uses the ranked queue to choose the next pages to inspect and decide whether an update is warranted.
- **Output:** A ranked opportunity/priority score with understandable reason codes where possible.
- **Action:** Review the highest-priority pages first; the human decides whether to refresh, improve CTR, investigate search visibility, or take no action.
- **Cost of a wrong call:** A false positive can waste editor/SEO time on a page that did not need attention. A false negative can leave a genuinely declining or underperforming page untreated. Because the team has limited review capacity, ranking quality matters.

This is **not just “train a model.”** A model is only useful if its ranked output improves a real prioritization decision compared with a simple baseline. I will therefore define the metric before modeling and compare any learned approach with a simple rule or baseline.


In [2]:
# State the decision grain and planned evaluation before modeling.
print("Decision grain: one content page per row in the starter dataset.")
print("Planned output: a ranked review queue.")
print("Planned primary evaluation: precision@K for the selected review capacity.")


Decision grain: one content page per row in the starter dataset.
Planned output: a ranked review queue.
Planned primary evaluation: precision@K for the selected review capacity.


## 3. Quick look at the data (2-3 real numbers)

The starter data contains **30,000 content items across 44 columns**. In the starter pipeline, the top 50 items have a **34.0% declining rate** under the starter's current-window proxy label. The observed CTR also differs substantially by position tier: **0.3548% for `page_1` versus 0.0554% for `deep`** among pages with at least 100 impressions.

These numbers make the lane worth investigating: there is a large enough set of pages to prioritize, a meaningful amount of decline in the current proxy outcome, and clear differences in search-performance measurements across position tiers. They are evidence for further investigation, not proof that refreshing a page will cause its performance to improve.


In [3]:
# Supporting numbers from the starter dataset / starter analysis.
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print("Top-50 declining rate (starter proxy): 0.340")
print("\nCTR by position tier (pages with impressions >= 100):")
print("page_1    0.3548")
print("deep      0.0554")


Rows: 30,000
Columns: 44
Top-50 declining rate (starter proxy): 0.340

CTR by position tier (pages with impressions >= 100):
page_1    0.3548
deep      0.0554


## 4. Careful words: what I can and can't claim

### What I can claim

- I can describe **observed associations and measured differences** in the starter data.
- I can test whether a ranking method gives better **decision-support** performance than a simple baseline on a defined validation split.
- If I later build a future-window target, I can evaluate whether earlier signals are useful for ranking or predicting that observed future outcome.

### What I cannot claim yet

- I cannot claim that refreshing a page **causes** traffic, CTR, or rankings to improve from this dataset alone.
- I cannot claim that the starter proxy label proves a future decline. The starter label is derived from the current `trend_direction`, so it is a proxy rather than an ideal future-looking target.
- I cannot claim to be “predicting Google” or guarantee a business outcome from a model score.
- I will not use `trend_direction` or `trend_pct` as model features when they define the starter label, and I will treat client/content IDs as identifiers rather than predictive features.

The project will stay focused on **decision support**: evidence → ranking → human review → action.


In [4]:
print("Self-check: target and features will be reviewed for leakage before modeling.")
print("Starter label is treated as a proxy, not as causal proof.")


Self-check: target and features will be reviewed for leakage before modeling.
Starter label is treated as a proxy, not as causal proof.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries are used
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`

**Next step:** keep this lane provisional, then use later weeks to test the signal, define a stronger future-looking target if needed, build a baseline, and validate whether the ranking actually improves the decision.
